In [15]:
import matplotlib.pyplot as plt
import numpy as np
from mazelib import Maze
from mazelib.generate.BacktrackingGenerator import BacktrackingGenerator
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import pdb

In [2]:
ACTIONS = np.array([
    (-1, 0), # down
    (1, 0),  # up
    (0, -1), # left
    (0, 1),  # right
], dtype=np.int32)

In [3]:
def generate_one(cells_h: int, cells_w: int, seed: int):
    np.random.seed(seed)
    maze = Maze(seed = seed)
    maze.generator = BacktrackingGenerator(cells_h, cells_w)
    maze.generate()

    grid = np.array(maze.grid, dtype=np.int8)
    start = (1, 1)
    goal = (grid.shape[0] - 2, grid.shape[1] - 2)

    grid[start] = 0
    grid[goal] = 0

    return grid, start, goal

In [4]:
def plot_maze(grid: np.ndarray, start: tuple[int, int], goal: tuple[int, int]):
    h, w = grid.shape
    img = np.zeros((h, w, 3), dtype=np.float32)
    img[grid==1] = [0.08, 0.08, 0.08]
    img[grid==0] = [0.97, 0.97, 0.97]
    img[start] = [0.20, 0.75, 0.20]
    img[goal] = [0.85, 0.20, 0.20]

    plt.figure(figsize=(5, 5))
    plt.imshow(img, interpolation="nearest")
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()
    plt.show()

In [5]:
class MazeEnv:
    def __init__(self, grid: np.ndarray, start=(1, 1), goal=None):
        self.grid = grid
        self.h, self.w = grid.shape
        self.start = start
        self.goal = goal if goal is not None else (self.h -2, self.w - 2)

    def reset(self):
        return self.start

    def is_open(self, r, c):
        return 0 <= r < self.h and 0 <= c < self.w and self.grid[r, c] == 0

    def step(self, state, action):
        r, c = state
        dr, dc = ACTIONS[action]
        nr, nc = r + dr, c + dc

        if not self.is_open(nr, nc):
            nr, nc = r, c

        next_state = (int(nr), int(nc))
        done = next_state == self.goal
        reward = 1 if done else 0
        return next_state, reward, done

In [6]:
class MLPPolicyValue(nn.Module):
    def __init__(self, H: int, W: int, in_channels: int=3, hidden: int=512):
        super().__init__()
        self.H = H
        self.W = W
        self.in_channels = in_channels

        obs_dim = in_channels * H * W

        self.fc1 = nn.Linear(obs_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)

        self.pi = nn.Linear(hidden, 4)
        self.v = nn.Linear(hidden, 1)

        nn.init.orthogonal_(self.fc1.weight, gain=1.0)
        nn.init.orthogonal_(self.fc2.weight, gain=1.0)
        nn.init.orthogonal_(self.pi.weight, gain=0.01)
        nn.init.orthogonal_(self.v.weight, gain=1.0)

        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)
        nn.init.zeros_(self.pi.bias)
        nn.init.zeros_(self.v.bias)

    def forward(self, obs: torch.Tensor):
        B = obs.shape[0]
        x = obs.view(B, -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        logits = self.pi(x) # (B, 4)
        value = self.v(x).squeeze(-1) # (B,)
        return logits, value        

In [7]:
def build_obs(grid: torch.Tensor, agent_rc: tuple[int, int], goal_rc: tuple[int, int]):
    """
    grid: (H, W) int/bool/float
    return obs: (3, H, W) float 32
    """
    H, W = grid.shape
    wall = grid.float()

    agent = torch.zeros((H, W), dtype=torch.float32)
    goal = torch.zeros((H, W), dtype=torch.float32)
    ar, ac = agent_rc
    gr, gc = goal_rc
    agent[ar, ac] = 1.0
    goal[gr, gc] = 1.0

    obs = torch.stack([wall, agent, goal], dim=0)
    return obs

In [8]:
class RolloutBuffer:
    def __init__(self):
        self.buffer = list()

    def store(self, transition):
        self.buffer.append(transition)

    def sample(self):
        states, actions, rewards, next_states, dones = map(np.array, zip(*self.buffer))
        self.buffer.clear()
        return (
            torch.FloatTensor(states),
            torch.FloatTensor(actions),
            torch.FloatTensor(rewards).unsqueeze(1),
            torch.FloatTensor(next_states),
            torch.FloatTensor(dones).unsqueeze(1)
        )

    @property
    def size(self):
        return len(self.buffer)

In [31]:
class PPO:
    def __init__(self, 
                 policy_value_network:nn.Module,
                 n_steps=960, 
                 n_epochs=10,
                 batch_size=64,
                 lr=0.0003,
                 gamma=0.99,
                 lmda=0.95,
                 clip_ratio=0.2,
                 vf_coef=1.0,
                 ent_coef=0.01,):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.policy_value_network = policy_value_network
        self.n_steps = n_steps
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.lr = lr
        self.gamma = gamma
        self.lmda = lmda
        self.clip_ratio=clip_ratio
        self.vf_coef=vf_coef
        self.ent_coef=ent_coef

        self.optimizer = torch.optim.Adam(self.policy_value_network.parameters(), lr=self.lr)

        self.buffer = RolloutBuffer()

    @torch.no_grad()
    def act(self, state: torch.Tensor, training=True):
        self.policy_value_network.train(training)
        logits, _ = self.policy_value_network(state.view(1, -1))
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        return action.item()

    def learn(self):
        self.policy_value_network.train()
        states, actions, rewards, next_states, dones = self.buffer.sample()


        states = states.to(self.device)
        next_states = next_states.to(self.device)
        actions = actions.to(self.device).long().view(-1)
        rewards = rewards.to(self.device).view(-1)
        dones = dones.to(self.device).view(-1)
        
        
        logits, values = self.policy_value_network(states)
        _, next_values = self.policy_value_network(next_states)

        
        with torch.no_grad():
            delta = rewards + (1 - dones) * self.gamma * next_values - values
            adv = torch.clone(delta)
            ret = torch.clone(rewards)
            ret[-1] += (1 - dones[-1]) * self.gamma * next_values[-1] 

            for t in reversed(range(len(rewards) - 1)):
                adv[t] += (1 - dones[t]) * self.gamma * self.lmda * adv[t + 1]
                ret[t] += (1 - dones[t]) * self.gamma * ret[t + 1]

            dist = torch.distributions.Categorical(logits=logits)
            log_probs_old = dist.log_prob(actions)

        dataset = TensorDataset(states, actions, ret, adv, log_probs_old)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        for e in range(self.n_epochs):
            value_losses = []
            policy_losses = []
            entropy_bonuses = []

            for batch in loader:
                state_, action_, ret_, adv_, log_prob_old_ = batch
                state_ = state_.to(self.device)
                action_ = action_.to(self.device).view(-1)
                adv_ = adv_.to(self.device).view(-1)
                log_prob_old_ = log_prob_old_.to(self.device).view(-1)
                logits_, values = self.policy_value_network(state_)
                value_loss = F.mse_loss(values, ret_)

                dist = torch.distributions.Categorical(logits=logits_)
                log_prob = dist.log_prob(action_)

                ratio = (log_prob - log_prob_old_).exp()

                surr1 = adv_ * ratio
                surr2 = adv_ * torch.clamp(ratio, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio)

                policy_loss = -torch.min(surr1, surr2).mean()
                entropy_bonus = dist.entropy().mean()

                loss = policy_loss + self.vf_coef * value_loss - self.ent_coef * entropy_bonus
                self.optimizer.zero_grad()

                loss.backward()
                self.optimizer.step()

                value_losses.append(value_loss.item())
                policy_losses.append(policy_loss.item())
                entropy_bonuses.append(entropy_bonus.item())

        result = {'policy_loss': float(np.mean(policy_losses)), 
                  'value_loss': float(np.mean(value_losses)), 
                  'entropy_bonus': float(np.mean(entropy_bonuses))}
        return result
                
    def step(self, transition):
        result = None
        self.buffer.store(transition)
        if self.buffer.size >= self.n_steps:
            result = self.learn()

        return result

In [32]:
def evaluate(grid, start, goal, agent: PPO, eval_iterations, max_steps=350):
    grid_t = torch.from_numpy(grid).float()
    start_t = torch.tensor(start, dtype=torch.long)
    goal_t = torch.tensor(goal, dtype=torch.long)
    env = MazeEnv(grid, start, goal)
    scores = []
    for i in range(eval_iterations):
        state = build_obs(grid_t, start_t, goal_t)
        position = start
        score = 0
        for t in range(max_steps):
            action = agent.act(state, training=False)
            next_position, reward, done = env.step(position, action)
            score += reward
            next_state_t = torch.tensor(next_position, dtype=torch.long)
            state = build_obs(grid_t, next_position, goal_t)
        scores.append(score)

    return round(np.mean(scores), 4)

In [ ]:
max_iterations = 100000
eval_intervals = 1000
eval_iterations = 10
max_steps = 350
step = 0

grid, start, goal = generate_one(3, 3, 1)
grid_t = torch.from_numpy(grid).float()
start_t = torch.tensor(start, dtype=torch.long)
goal_t = torch.tensor(goal, dtype=torch.long)
env = MazeEnv(grid, start, goal)

policy_value_network = MLPPolicyValue(grid_t.shape[0], grid_t.shape[1])
agent = PPO(policy_value_network=policy_value_network)

logger = []


position_t = start_t
position = start
state = build_obs(grid_t, start_t, goal_t)

for t in tqdm(range(1, max_iterations + 1)):
    action = agent.act(state)
    
    next_position, reward, done = env.step(position, action)
    next_position_t = torch.tensor(next_position, dtype=torch.long)

    state = build_obs(grid_t, position_t, goal_t)
    next_state = build_obs(grid_t, next_position_t, goal_t)
    result = agent.step((state, action, reward, next_state, done))

    state = next_state
    position = next_position
    position_t = next_position_t
    
    if result is not None:
        logger.append([t, 'policy_loss', result['policy_loss']])
        logger.append([t, 'value_loss', result['value_loss']])
        logger.append([t, 'entropy_bonus', result['entropy_bonus']])

    if done or step == 350:
        step = 0
        position_t = start_t
        state = build_obs(grid_t, start_t, goal_t)

    if t % eval_intervals == 0:
        score = evaluate(grid, start, goal, agent, eval_iterations, max_steps=350)
        logger.append([t, 'Avg return', score])
        print(f"[{t}/{max_iterations}] score: {score}")

    step += 1

  2%|▉                                                       | 1610/100000 [00:01<01:41, 967.84it/s]

[1000/100000] score: 0.0


  3%|█▍                                                      | 2526/100000 [00:02<01:55, 840.67it/s]

[2000/100000] score: 0.0


  3%|█▉                                                      | 3457/100000 [00:04<01:58, 812.14it/s]

...

 72%|███████████████████████████████████████▊               | 72395/100000 [01:43<00:39, 702.88it/s]

[72000/100000] score: 0.0


 73%|████████████████████████████████████████▍              | 73486/100000 [01:45<00:34, 763.21it/s]

[73000/100000] score: 0.0


 74%|███████████████████████████████████████▉              | 73855/100000 [01:45<00:25, 1013.32it/s]

In [ ]:
agent.n_epochs

In [ ]:
agent.buffer.buffer

In [ ]:
agent.buffer.buffer

In [ ]:
print(action)

In [ ]:
evaluate(grid, start, goal, agent, 10)

In [ ]:
grid, start, goal = generate_one(3, 3, 1)
env = MazeEnv(grid, start, goal)

grid_t = torch.from_numpy(grid).float()
start_t = torch.tensor(start, dtype=torch.long)
goal_t = torch.tensor(goal, dtype=torch.long)
obs = build_obs(grid_t, start_t, goal_t)
policy_value = MLPPolicyValue(grid_t.shape[0], grid_t.shape[1])
logits, value = policy_value(obs.view(1, -1))

dist = torch.distributions.Categorical(logits=logits)
action = dist.sample()
next_state, reward, done = env.step(start, action)

plot_maze(grid, next_state, goal)

In [ ]:
agent = PPO(policy_value_network=policy_value)